In [0]:
%pip install gspread google-auth
dbutils.library.restartPython()

In [0]:
import gspread
from google.oauth2.service_account import Credentials
import json

KEY_FILE_PATH = "/Workspace/Users/pakhei_tsang@next.co.uk/advance-mantis-398714-2168c9162641.json"

with open(KEY_FILE_PATH, "r") as f:
    creds_dict = json.load(f)

creds = Credentials.from_service_account_info(
    creds_dict,
    scopes=["https://www.googleapis.com/auth/spreadsheets"]
)
gc = gspread.authorize(creds)

SHEET_ID = "1uiv27J0YPDWqSVuN-QLpS-fOcfxAjewK5ajTd3aA8vI"
sh = gc.open_by_key(SHEET_ID)

print("Connected to:", sh.title)

In [0]:
df = (spark.read
      .format("delta")
      .load("abfss://landing@whsanalyticsdlsprodeuw.dfs.core.windows.net/streaming/landing_bonushub_event_parsed/delta/"))

df.createOrReplaceTempView("landing_bonus_hub_event_parsed")

print("Row count:", df.count())
df.printSchema()

In [0]:
# Online Picking carries its zone inside a location string rather than in an
# attribute of its own: "Location: BA356C" decodes as zone B, aisle A, bay 356,
# level C. So a zone filter is a test on the FIRST CHARACTER after the
# "Location:" prefix, not a match on a whole attribute value - which is why
# matching on 'Zone: D' returned nothing at all.
#
# substring(x, 10) drops the prefix; trim() then copes with the space after the
# colon being there or not, and the first character of what is left is the zone.
def _zone_pred(*zones):
    _z = ", ".join(f"'{z}'" for z in zones)
    return ("AND EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE 'Location:%' AND "
            f"upper(substring(trim(substring(x, 10)), 1, 1)) IN ({_z}))")


reports = [
    # attributes: this report's PAYLOAD_ATTRIBUTES reach the Data tab's
    # Attribute column, and its rows split by attribute set as a result.
    {"tab": "Data", "name": "D.Analysis - OSR PiE",
     "area": "('Pick','Pie Station')", "event": "('COUN','MSKU','PSKU','RSIN')", "extra": "",
     "attributes": True},

    {"tab": "Data", "name": "D.Analysis - OSR Topup",
     "area": "('Topup','TopUp','PiOrQi')", "event": "('QSIN','TARS','TPUT')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - E3 Packing",
     "area": "('E3 - Packing')", "event": "('PackingParcelCompleteEvent','PackingItemScannedEvent')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - Parcel Induct",
     "area": "('Parcel Induct')", "event": "('APAR','PIND','SPAR','UPAR')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - Parcel Sortation",
     "area": None, "event": "('ParcelSortedToSack','SackMappedToPosition','SackUnMappedFromPosition')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - Inbound Decanting",
     "area": "('Inbound Decanting')", "event": "('DECN','TOPR')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - OSR Decanting",
     "area": "('OSR Decanting')", "event": "('ODEC','OTOP')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - BCR Inducting",
     "area": "('Induct from E1/E2')", "event": "('SPOS')",
     "extra": "AND EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE '%RET%')"},

    {"tab": "Data", "name": "D.Analysis - E1/E2 Inducting",
     "area": "('Induct from E1/E2')", "event": "('SPOS')",
     "extra": "AND EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE '%PIE4EDW%')"},

    # No event filter: every event type in this area earns standard hours.
    # Only PackingItemScannedEvent counts towards VOLUME - a separate concern,
    # handled by vol_events in PROC_AREAS.
    {"tab": "Data", "name": "D.Analysis - Sorter 6 Packing",
     "area": "('Sorter 6 - Packing')", "event": None, "extra": ""},

    # Online Picking is ONE source split three ways by zone: same warehouse,
    # same area code, different zones decoded out of the location string. No
    # event filter, so every event type earns standard hours; only PickItemEvent
    # counts towards volume (see vol_events).
    #
    # attributes: "zone_aisle" puts "Zone B, Aisle A" on the Data tab rather
    # than the raw location. Grouping follows the Attribute column, so these
    # reports collapse to one row per operator per event type per window per
    # zone-and-aisle. Carrying the full location instead would group by bay and
    # level too - a row for every item picked."
    #
    # A row somehow carrying two locations in different zones would feed both
    # reports; the array + explode tagging below keeps that working rather than
    # silently picking one.
    {"tab": "Data", "name": "D.Analysis - Online Picking - Drive",
     "area": "('Online Picking')", "event": None,
     "extra": _zone_pred('D', 'F', 'Q', 'S', 'T', 'V'),
     "attributes": "zone_aisle"},
    {"tab": "Data", "name": "D.Analysis - Online Picking - Way",
     "area": "('Online Picking')", "event": None,
     "extra": _zone_pred('A', 'B', 'E', 'G', 'J', 'L', 'X', 'W', 'R'),
     "attributes": "zone_aisle"},
    {"tab": "Data", "name": "D.Analysis - Online Picking - E3",
     "area": "('Online Picking')", "event": None,
     "extra": _zone_pred('H', 'C'),
     "attributes": "zone_aisle"},
]

COLUMNS = ['Date', 'Hour', 'PAYLOAD_BONUSCODE', 'PAYLOAD_EVENTTYPE', 'Attribute',
           'Total_Quantity', 'Total_StandardHours', 'Total_SMV',
           'Week', 'Date Time Range', 'Report Name']

In [0]:
# --- Std Hours divisor: read from Front!C2 of the linked spreadsheet (the same
#     value the Apps Script side used before processing moved into Databricks).
#     Accepts a percentage cell ("96.40%") or a bare number. `sh` is the
#     spreadsheet handle from the connection cell above. ---
_div_raw = (sh.worksheet("Front").acell("C2").value or "").strip()
if not _div_raw:
    raise ValueError("Front!C2 is empty - cannot determine the Std Hours divisor.")
divisor = float(_div_raw.rstrip('%')) / 100 if _div_raw.endswith('%') else float(_div_raw)
if divisor == 0:
    divisor = 1.0
print(f"Divisor (from Front!C2): {divisor}")

# --- Work-area pivot config: the column ORDER of Processed Data (15mins), each
#     area's volume event type(s), and the short label its volume column carries.
#     Single source of truth - the header row, the data rows and the clear range
#     are all derived from this list, so adding an area here is the only edit
#     the tab's layout needs. ---
PROC_AREAS = [
    {"report": "D.Analysis - OSR PiE",           "label": "PiE",               "vol_events": {"MSKU", "PSKU"}},
    {"report": "D.Analysis - OSR Topup",         "label": "Top Up",            "vol_events": {"TPUT"}},
    {"report": "D.Analysis - E3 Packing",        "label": "E3 Packing",        "vol_events": {"PackingItemScannedEvent"}},
    {"report": "D.Analysis - Parcel Sortation",  "label": "Parcel Sortation",  "vol_events": {"ParcelSortedToSack"}},
    {"report": "D.Analysis - Parcel Induct",     "label": "Parcel Induct",     "vol_events": {"SPAR"}},
    {"report": "D.Analysis - Inbound Decanting", "label": "Inbound Decanting", "vol_events": {"DECN"}},
    {"report": "D.Analysis - OSR Decanting",     "label": "OSR Decanting",     "vol_events": {"ODEC"}},
    {"report": "D.Analysis - BCR Inducting",     "label": "BCR Inducting",     "vol_events": {"SPOS"}},
    {"report": "D.Analysis - E1/E2 Inducting",   "label": "E1/E2 Inducting",   "vol_events": {"SPOS"}},
    {"report": "D.Analysis - Sorter 6 Packing",  "label": "Sorter 6 Packing",  "vol_events": {"PackingItemScannedEvent"}},
    {"report": "D.Analysis - Online Picking - Drive", "label": "Online Picking - Drive", "vol_events": {"PickItemEvent"}},
    {"report": "D.Analysis - Online Picking - Way",   "label": "Online Picking - Way",   "vol_events": {"PickItemEvent"}},
    {"report": "D.Analysis - Online Picking - E3",    "label": "Online Picking - E3",    "vol_events": {"PickItemEvent"}},
]

# The header row is written from PROC_AREAS as well, rather than maintained by
# hand on the tab. The dashboard locates every column by its header NAME, so a
# header that disagrees with the data underneath it is the one mistake that
# mis-maps the whole tab silently - and inserting a work area shifts every
# column after it. Deriving both from this list makes that disagreement
# impossible.
PROC_HEADER = (["Date", "Hour", "BONUS"]
               + [a["report"] for a in PROC_AREAS]
               + ["Sum of Std hrs"]
               + [f"Volume - {a['label']}" for a in PROC_AREAS])


def _col_a1(n):
    """1-based column number -> A1 letters (24 -> 'X')."""
    s = ""
    while n:
        n, r = divmod(n - 1, 26)
        s = chr(65 + r) + s
    return s


PROC_LAST_COL = _col_a1(len(PROC_HEADER))


In [0]:
try:
    ws = sh.worksheet("Data")
except gspread.exceptions.WorksheetNotFound:
    ws = sh.add_worksheet(title="Data", rows=2000, cols=len(COLUMNS))
    ws.update('A1', [COLUMNS])
    print("Created 'Data' tab with header")

In [0]:
# ============================================================
# WINDOW SELECTION - derived from BOTH the Data tab (which 15-min windows are
# already recorded) AND wall-clock (which windows are old enough to be safe
# to pull, i.e. BonusHub events have had time to land in the Delta table).
#
# The "newest safe" window is anchored to the current (possibly still-open)
# 15-min grid boundary - floor(now) - then stepped back one extra full grid
# step (SAFETY_MARGIN_MINUTES = 15), rather than computed from elapsed
# wall-clock time directly off `now`. This makes the result STABLE across an
# entire 15-min grid cell: a run scheduled at :01 that's delayed by Job
# Cluster cold-start and doesn't actually execute this cell until :11 (or
# anywhere up to :14) computes the exact SAME window as if it had run right
# on time at :01 - instead of drifting forward the later it happens to start.
#   e.g. floor(now) = 10:00 -> newest safe window = 09:30-09:45,
#        whether this cell actually runs at 10:01, 10:11, or 10:14.
# Minimum staleness is 15 min (cell runs right at the grid line); worst case
# (cell runs just under 15 min late) is just under 30 min. If a run is
# delayed PAST the next grid line, floor(now) simply advances and the window
# shifts forward with it - nothing is silently skipped, because any window a
# late run doesn't reach still shows up as a gap below and gets caught on a
# later run.
#
# Missing windows are found by diffing the full set of expected window-end
# times - walking back from the newest safe window, up to LOOKBACK_WINDOWS -
# against the "Date Time Range" values already written to Data. Using the
# full recorded set (not just a single max-watermark) means a hole anywhere
# in that horizon gets noticed and refilled, not just a late-starting run.
#
# Catch-up windows are processed NEWEST-FIRST (closest to now), each run
# working backwards through any backlog up to MAX_CATCHUP windows, so the
# live dashboard gets fresh data before older backlog is filled in.
#
# The floor/margin math is anchored to the job's SCHEDULED trigger time (via
# the trigger_time_iso parameter, see below), not to whenever this cell
# actually executes - so an arbitrarily long cold start no longer shifts
# which window gets selected. Without that parameter set, it falls back to
# real wall-clock time at execution, same behavior as before.
#
# Schedule the job with a minimal offset past each grid line (e.g. :01/:16/
# :31/:46) - it no longer needs tuning to outlast typical cold-start lag,
# since the margin above already absorbs it.
# ============================================================
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

uk = ZoneInfo("Europe/London")
utc = ZoneInfo("UTC")

dbutils.widgets.text("max_catchup_windows", "8", "Max Catch-up Windows")
MAX_CATCHUP = int(dbutils.widgets.get("max_catchup_windows"))

dbutils.widgets.text("lookback_windows", "96", "Lookback Windows (gap-scan horizon)")
LOOKBACK_WINDOWS = int(dbutils.widgets.get("lookback_windows"))

SAFETY_MARGIN_MINUTES = 15  # newest safe window ends this many minutes before
                             # the current 15-min grid boundary (floor(now)) -
                             # one full extra grid step back. See comment
                             # above for why this beats an elapsed-time offset.

def _parse_range_end_utc(dtr):
    try:
        end_str = dtr.split(' - ')[1].strip()
        end_local = datetime.strptime(end_str, '%d/%m/%Y %H:%M').replace(tzinfo=uk)
        return end_local.astimezone(utc)
    except Exception:
        return None

def _week_val(date_local):
    epoch_ref = datetime(2025, 12, 14).date()
    return ((date_local - epoch_ref).days // 7 + 46) % 52

# Anchor to the job's actual SCHEDULED trigger time, not whenever this cell
# happens to execute. A Job Cluster cold start can delay execution by 15+
# minutes past the nominal trigger, and datetime.now() only ever sees that
# later real clock - it has no way to know what time the job was *meant* to
# fire. Wire the job's trigger time in via a parameter using Databricks'
# dynamic value reference: set a job parameter named trigger_time_iso to
# {{job.trigger.time.iso_datetime}} in the job's Parameters config (Databricks
# resolves this to the scheduled fire time for schedule-triggered runs). If
# it's absent/unparseable (manual run, ad-hoc test, older job config), this
# falls back to real wall-clock time, same as before.
dbutils.widgets.text("trigger_time_iso", "", "Job Trigger Time (ISO, set via {{job.trigger.time.iso_datetime}})")
_trigger_raw = dbutils.widgets.get("trigger_time_iso").strip()
now_utc = datetime.now(utc)
if _trigger_raw:
    try:
        _anchor = datetime.fromisoformat(_trigger_raw.replace('Z', '+00:00'))
        anchor_utc = _anchor.astimezone(utc) if _anchor.tzinfo else _anchor.replace(tzinfo=utc)
    except Exception:
        anchor_utc = now_utc
else:
    anchor_utc = now_utc

floor_epoch = (int(anchor_utc.timestamp()) // 900) * 900
floor_utc = datetime.fromtimestamp(floor_epoch, utc)
newest_safe_end_utc = floor_utc - timedelta(minutes=SAFETY_MARGIN_MINUTES)

# Never reach back before today's local midnight, give or take a small grace
# period. The last two windows of a day can only be pulled after midnight (a
# window ending 00:00 is not safe to read until 00:15), so the scan has to be
# allowed to reach back slightly past the boundary or those windows would
# never be fetched at all.
#
# This is now purely a floor on how far back a GENUINELY UNFETCHED window may
# be chased. It is no longer doing the work of deciding what has already been
# handled - the coverage record above does that, and it does not forget a
# window just because Archive.js moved its rows.
PRE_MIDNIGHT_GRACE_MINUTES = 30

today_local_date = anchor_utc.astimezone(uk).date()
start_of_today_utc = datetime(today_local_date.year, today_local_date.month, today_local_date.day, tzinfo=uk).astimezone(utc)
earliest_allowed_end_utc = start_of_today_utc - timedelta(minutes=PRE_MIDNIGHT_GRACE_MINUTES)

# --- Coverage record ------------------------------------------------------
# Which windows have already been fetched is recorded on its own tab, rather
# than inferred from what is currently sitting on the Data tab.
#
# Data is not a record of what has been fetched. Archive.js trims it to today
# at 01:00 and dailyDataCleanup prunes it at 04:00, so the last windows of
# yesterday - 23:30-23:45 and 23:45-00:00, which can only be pulled after
# midnight - are fetched, written, archived, and then read back as "missing"
# on the very next run. They get pulled again, duplicating rows that already
# live in the archive, and dailyDataCleanup then has to delete them. That
# fetch/archive/refetch/delete loop ran every day.
#
# A record that survives both jobs breaks the loop at its source: a window is
# fetched exactly once, whatever later happens to the rows it produced.
STATE_TAB = "Pipeline State"
STATE_HEADER = ["Window End (UTC)", "Date Time Range", "Rows Written", "Recorded At (UTC)"]
STATE_KEEP_DAYS = 7            # how much history to keep; pruned below
STATE_PRUNE_MIN_ROWS = 96      # a full day's worth - keeps this to ~one write a day

def _state_parse(s):
    try:
        return datetime.strptime(s.strip(), "%Y-%m-%d %H:%M:%S").replace(tzinfo=utc)
    except Exception:
        return None

try:
    state_ws = sh.worksheet(STATE_TAB)
    state_rows = state_ws.get_all_values()
except gspread.exceptions.WorksheetNotFound:
    state_ws = sh.add_worksheet(title=STATE_TAB, rows=1000, cols=len(STATE_HEADER))
    state_ws.update('A1', [STATE_HEADER])
    state_rows = []

covered_ends = set()
for _r in state_rows[1:]:
    if _r and _r[0].strip():
        _end = _state_parse(_r[0])
        if _end:
            covered_ends.add(_end)

# One-time seed. With no state yet - first run after this change, or the tab
# having been deleted - fall back to what the Data tab shows, so today's
# already-written windows are not all pulled a second time. From the next run
# on, the state tab is the only thing consulted.
if not covered_ends:
    for _r in ws.get_all_values()[1:]:
        if len(_r) > 8 and _r[8].strip():
            _end = _parse_range_end_utc(_r[8].strip())
            if _end:
                covered_ends.add(_end)
    print(f"No coverage state yet - seeded {len(covered_ends)} window(s) from the Data tab.")

# Keep the record from growing without bound - pruned by DATE, not by position
# on the sheet.
#
# "Keep the last N rows" sounds equivalent and is not. Rows land in the order
# they are written, so backfilling older days appends entries whose WINDOWS are
# old at the BOTTOM of the tab. A position-based prune keeps those and evicts
# genuine recent coverage that happened to be written earlier - and if an
# evicted window is still inside today's boundary, the next run re-fetches it
# and lays duplicate rows on the Data tab. The window's date is the property
# that actually decides whether an entry still matters.
#
# Rows whose Window End cannot be parsed are kept rather than dropped: they
# cannot mark anything covered, so they are harmless, and silently destroying
# rows nobody can read is not a trade worth making. If they ever accumulate,
# the count below says so.
_cutoff = anchor_utc - timedelta(days=STATE_KEEP_DAYS)
_fresh, _unreadable, _stale = [], [], 0
for _r in state_rows[1:]:
    _end = _state_parse(_r[0]) if (_r and _r[0].strip()) else None
    if _end is None:
        _unreadable.append(_r)
    elif _end >= _cutoff:
        _fresh.append(_r)
    else:
        _stale += 1

if _stale >= STATE_PRUNE_MIN_ROWS:
    # Write first, then clear the tail - same reason as the Processed Data
    # rebuild: never leave the tab empty or half-written between two calls.
    _kept = _unreadable + _fresh
    _old_last = len(state_rows)
    _new_last = len(_kept) + 1
    state_ws.update('A1', [STATE_HEADER] + _kept, value_input_option='RAW')
    if _old_last > _new_last:
        state_ws.batch_clear([f"A{_new_last + 1}:D{_old_last}"])
    print(f"Pruned coverage state: dropped {_stale} window(s) older than "
          f"{STATE_KEEP_DAYS} days, kept {len(_fresh)}"
          + (f" (+{len(_unreadable)} unreadable, left in place)" if _unreadable else ""))

missing_ends = []
cursor = newest_safe_end_utc
for _ in range(LOOKBACK_WINDOWS):
    if cursor <= earliest_allowed_end_utc:  # boundary itself is excluded, not just anything below it
        break
    if cursor not in covered_ends:
        missing_ends.append(cursor)
    cursor -= timedelta(minutes=15)

windows = []
for w_end_utc in missing_ends[:MAX_CATCHUP]:
    w_start_utc = w_end_utc - timedelta(minutes=15)
    w_start_local = w_start_utc.astimezone(uk)
    w_end_local = w_end_utc.astimezone(uk)
    windows.append({
        "start_utc": w_start_utc.strftime("%Y-%m-%d %H:%M:%S"),
        "end_utc":   w_end_utc.strftime("%Y-%m-%d %H:%M:%S"),
        "week_val":  _week_val(w_start_local.date()),
        "date_time_range": f"{w_start_local.strftime('%d/%m/%Y %H:%M')} - {w_end_local.strftime('%d/%m/%Y %H:%M')}",
    })

print(f"Actual execution time (UTC): {now_utc}")
print(f"Anchor used for window calc (UTC): {anchor_utc}" + ('  [from job trigger param]' if _trigger_raw else '  [wall-clock fallback - no trigger param set]'))
print(f"Current grid boundary floor(anchor) (UTC): {floor_utc}")
print(f"Newest safe window end (UTC): {newest_safe_end_utc}")
print(f"Start of today, local midnight (UTC): {start_of_today_utc}")
print(f"Earliest allowed window end, with grace (UTC): {earliest_allowed_end_utc}")
print(f"Missing windows in lookback horizon ({LOOKBACK_WINDOWS}): {len(missing_ends)}")
print(f"Windows to process this run (newest first): {len(windows)}")
for _w in windows:
    print(f"  {_w['date_time_range']}")
if len(missing_ends) > MAX_CATCHUP:
    print(f"WARNING: backlog exceeds max_catchup_windows ({MAX_CATCHUP}); "
          f"{len(missing_ends) - MAX_CATCHUP} older window(s) still pending — will catch up on subsequent runs.")
if not windows:
    print("No new windows to process.")


In [0]:
import pandas as pd
from datetime import datetime, timezone

# ============================================================
# CONSOLIDATED FETCH + BATCHED APPEND
# One Delta scan for the WHOLE run (all target windows x all reports), instead
# of len(windows) x len(reports) separate spark.sql().toPandas() calls, and ONE
# ws.append_rows() instead of one per (window, report).
#
# Every report reads the same table with the same warehouse filter and the same
# GROUP BY - they differ only by area code, event type and (for the two SPOS
# reports) a PAYLOAD_ATTRIBUTES predicate. So each raw event is tagged with the
# set of reports it belongs to (built as an array of CASE expressions, one per
# report, from the single-source `reports` config), then exploded. Using a
# *set* (array + explode) rather than a first-match CASE preserves the original
# semantics exactly: if a row satisfied two reports' filters it fed both, and it
# still does here - no reliance on the RET / PIE4EDW patterns being disjoint.
#
# Rows are written with value_input_option='RAW'. USER_ENTERED asks Sheets to
# INTERPRET every value, and column C is the bonus code: it turned "3E3" into
# the number 3000 (displayed back as "3.00E+03") and "2PM" into a time
# (displayed back as "14:00"). The same operator then appeared under two
# different names, and the whole NAME_TIME_MAP_ correction table on the Apps
# Script side existed only to undo that damage after the fact - too late for
# the rebuild below, which reads these rows back seconds after writing them.
# RAW stores what is sent, so nothing needs undoing. Numbers are therefore
# sent as real numbers rather than stringified, since RAW would keep a string
# a string.
#
# The scan is bounded to [min start, max end] of this run's windows for
# partition pruning, then filtered to the EXACT 15-min buckets we intend to fill
# (window_epoch IN ...). That last filter matters when the gap-scan picks
# NON-contiguous windows (e.g. 09:00 and 11:00 missing but 10:00 already
# covered): a plain range scan would otherwise re-pull 10:00 and append
# duplicate rows for an already-covered window.
# ============================================================

failures = []
total_rows_pushed = 0

if not windows:
    print("No windows to process; skipping fetch/append.")
else:
    # --- Map each 15-min window bucket (UTC epoch, floored to the grid) to its
    #     metadata. Windows are already 15-min aligned, so the floor below and
    #     FLOOR(unix_timestamp/900)*900 in SQL land on the same epoch. ---
    def _win_epoch(s):
        return int(datetime.strptime(s, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc).timestamp())

    # Sheets rejects NaN/Infinity outright (they are not valid JSON), and an
    # empty cell is the honest representation of "no figure" anyway.
    def _num(v):
        try:
            f = float(v)
        except (TypeError, ValueError):
            return ''
        if f != f or f in (float('inf'), float('-inf')):
            return ''
        return f

    def _int(v):
        try:
            return int(float(v))
        except (TypeError, ValueError):
            return ''

    win_by_epoch = {}
    for _w in windows:
        win_by_epoch[_win_epoch(_w['start_utc'])] = _w

    overall_start_utc = min(w['start_utc'] for w in windows)
    overall_end_utc = max(w['end_utc'] for w in windows)
    epoch_list = ", ".join(str(e) for e in sorted(win_by_epoch))

    # --- Turn one report's {area, event, extra} config into a standalone SQL
    #     boolean predicate (the `extra` field starts with "AND ", stripped). ---
    def _report_predicate(r):
        parts = []
        if r['area']:
            parts.append(f"PAYLOAD_AREACODE IN {r['area']}")
        # event is optional in the same way area already is: Sorter 6 Packing
        # earns standard hours from EVERY event type in its area, so it carries
        # no event filter. Volume stays restricted, via PROC_AREAS' vol_events.
        if r['event']:
            parts.append(f"PAYLOAD_EVENTTYPE IN {r['event']}")
        if not parts:
            raise ValueError(
                f"Report {r['name']!r} has neither an area nor an event filter - "
                "it would match every event in the warehouse."
            )
        extra = r['extra'].strip()
        if extra:
            if extra[:4].upper() == "AND ":
                extra = extra[4:].strip()
            parts.append(extra)
        return " AND ".join(f"({p})" for p in parts)

    preds = [(_report_predicate(r), r['name']) for r in reports]
    case_exprs = ",\n            ".join(f"CASE WHEN {pred} THEN '{name}' END" for pred, name in preds)
    match_any = " OR ".join(f"({pred})" for pred, _ in preds)

    # "Location: BA356C" decodes as zone B, aisle A, bay 356, level C. Online
    # Picking reports on zone and aisle, so the pair is derived here and the bay
    # and level are dropped - carrying the whole location would group by it, and
    # put a row on the Data tab for every item picked.
    # try_element_at, NOT element_at: under ANSI mode element_at raises
    # INVALID_ARRAY_INDEX_IN_ELEMENT_AT on an EMPTY array instead of returning
    # NULL, and an event carrying no Location attribute filters down to exactly
    # that. try_element_at returns NULL, which the CASE below already handles.
    _loc = ("trim(substring(try_element_at("
            "filter(PAYLOAD_ATTRIBUTES, x -> x LIKE 'Location:%'), 1), 10))")
    zone_aisle_sql = (
        # try_element_at yields NULL when an event carries no location at all, and
        # concat would propagate that, so the empty case is handled first.
        f"CASE WHEN {_loc} IS NULL OR length({_loc}) < 2 THEN '' "
        f"ELSE concat('Zone ', upper(substring({_loc}, 1, 1)), "
        f"', Aisle ', upper(substring({_loc}, 2, 1))) END"
    )

    # A report's `attributes` says WHAT its Attribute column holds: True for the
    # raw attribute list, or the name of a column derived in the CTE below.
    # Whichever it is, the value joins the GROUP BY for that report alone, so
    # its rows split by that value while every other report gets '' and groups
    # exactly as it did before.
    _attr_cases = []
    for _r in reports:
        _a = _r.get('attributes')
        if not _a:
            continue
        _expr = 'attrs' if _a is True else _a
        _attr_cases.append(f"WHEN report_name = '{_r['name']}' THEN {_expr}")
    if _attr_cases:
        attr_sel = "CASE " + " ".join(_attr_cases) + " ELSE '' END"
        # Repeated in GROUP BY rather than referenced by alias, which not every
        # Spark version accepts.
        attr_group = ",\n        " + attr_sel
    else:
        attr_sel = "''"
        attr_group = ""

    query = f"""
      WITH base AS (
        SELECT
          date_format(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP),'Europe/London'),'dd/MM/yyyy') AS Date,
          hour(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP),'Europe/London')) AS Hour,
          CAST(FLOOR(unix_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP)) / 900) * 900 AS BIGINT) AS window_epoch,
          -- Canonicalised here, at the point the value enters the pipeline.
          -- The bonus code is an identifier and the source is not consistent
          -- about case, so "mf5" and "MF5" arrived as two different people:
          -- two rows out of this GROUP BY, two rows on the Data tab, two rows
          -- in the pivot, two heads on the dashboard, and each with half the
          -- standard hours they had actually earned. Folding case in the SQL
          -- merges them before anything downstream can tell them apart.
          upper(trim(PAYLOAD_BONUSCODE)) AS bonus,
          PAYLOAD_EVENTTYPE AS eventtype,
          -- Flattened here so the Data tab gets one readable cell rather than a
          -- stringified array. concat_ws drops nulls and yields '' for an empty
          -- or absent array, which is what an event with no attributes should
          -- show.
          concat_ws(', ', PAYLOAD_ATTRIBUTES) AS attrs,
          -- Zone and aisle, decoded from the location attribute. See zone_aisle_sql.
          {zone_aisle_sql} AS zone_aisle,
          PAYLOAD_QUANTITY      AS qty,
          PAYLOAD_STANDARDHOURS AS std,
          PAYLOAD_SMV           AS smv,
          filter(array(
            {case_exprs}
          ), x -> x IS NOT NULL) AS report_names
        FROM landing_bonus_hub_event_parsed
        WHERE TRIM(PAYLOAD_WAREHOUSECODE) = 'X'
          AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) >= timestamp('{overall_start_utc}')
          AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) <  timestamp('{overall_end_utc}')
          AND ( {match_any} )
      )
      SELECT
        Date, Hour, window_epoch, bonus, eventtype,
        {attr_sel} AS attribute,
        report_name,
        SUM(qty) AS Total_Quantity,
        SUM(std) AS Total_StandardHours,
        SUM(smv) AS Total_SMV
      FROM base
      LATERAL VIEW explode(report_names) t AS report_name
      WHERE window_epoch IN ({epoch_list})
      GROUP BY Date, Hour, window_epoch, bonus, eventtype, report_name{attr_group}
    """

    report_order = {r['name']: i for i, r in enumerate(reports)}

    # --- Single scan. A query-level failure is all-or-nothing now (same
    #     templated SQL used to fail every report anyway); it's captured so the
    #     Failures tab still records it and cell 9 still raises. ---
    pdf = None
    try:
        pdf = spark.sql(query).toPandas()
        print(f"Consolidated query returned {len(pdf)} grouped rows "
              f"across {len(windows)} window(s) x {len(reports)} report(s).")
    except Exception as e:
        error_msg = f"{type(e).__name__}: {str(e)}"
        print(f"CONSOLIDATED QUERY FAILED - {error_msg}")
        failures.append({"report": "(consolidated query)", "error": error_msg,
                         "window": f"{overall_start_utc}..{overall_end_utc} UTC"})

    if pdf is not None:
        # --- Shape into the Data-tab COLUMNS order; note which windows had data. ---
        rows = []
        windows_with_data = set()
        for _, row in pdf.iterrows():
            ep = int(row['window_epoch'])
            w = win_by_epoch.get(ep)
            if w is None:
                continue  # safety: outside the target buckets
            windows_with_data.add(ep)
            hour_key = int(row['Hour']) if str(row['Hour']).strip().lstrip('-').isdigit() else 0
            rows.append({
                "sort": (ep, report_order.get(row['report_name'], 999), hour_key, str(row['bonus'])),
                "epoch": ep,
                "vals": [
                    str(row['Date']),
                    _int(row['Hour']),
                    # str(), and RAW above: the bonus code is an identifier, not
                    # a quantity. This is the cell the whole correction problem
                    # was about.
                    str(row['bonus']),
                    str(row['eventtype']),
                    str(row['attribute']),
                    _num(row['Total_Quantity']),
                    _num(row['Total_StandardHours']),
                    _num(row['Total_SMV']),
                    int(w['week_val']),
                    w['date_time_range'],
                    str(row['report_name']),
                ],
            })

        # --- One placeholder per EMPTY window so the gap-scan (cell 6) marks it
        #     covered and doesn't re-fetch it forever. cell 8 skips these (Hour
        #     == ''). One per window is enough (was one per report per window). ---
        for ep, w in win_by_epoch.items():
            if ep not in windows_with_data:
                rows.append({
                    "sort": (ep, 999, 0, ''),
                    "epoch": ep,
                    "vals": ['', '', '', '(no data this window)', '', '', '', '',
                             int(w['week_val']), w['date_time_range'], '(no data this window)'],
                })

        rows.sort(key=lambda d: d['sort'])
        values = [d['vals'] for d in rows]

        rows_per_window = {}
        for d in rows:
            rows_per_window[d['epoch']] = rows_per_window.get(d['epoch'], 0) + 1

        # --- ONE batched append (was up to len(windows) x len(reports) writes,
        #     which also brushed the Sheets 60-writes/min quota). ---
        if values:
            try:
                ws.append_rows(values, value_input_option='RAW', table_range='A1')
                total_rows_pushed = len(values)
                print(f"Appended {total_rows_pushed} rows in one batch.")

                # Recorded ONLY on a successful append. A window marked covered
                # that never actually landed would never be retried, so this
                # must not run speculatively - a failed append leaves the
                # windows unrecorded and the next run picks them up as gaps.
                _stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
                state_ws.append_rows(
                    [[win_by_epoch[ep]['end_utc'],
                      win_by_epoch[ep]['date_time_range'],
                      rows_per_window.get(ep, 0),
                      _stamp] for ep in sorted(win_by_epoch)],
                    value_input_option='RAW', table_range='A1')
            except Exception as e:
                error_msg = f"{type(e).__name__}: {str(e)}"
                print(f"BATCH APPEND FAILED - {error_msg}")
                failures.append({"report": "(sheet append)", "error": error_msg,
                                 "window": f"{len(windows)} window(s)"})

        # --- Per-window x report breakdown for the run log. ---
        counts = pdf.groupby(['window_epoch', 'report_name']).size()
        for ep in sorted(win_by_epoch):
            w = win_by_epoch[ep]
            print(f"\n=== Window: {w['date_time_range']} ===")
            any_rows = False
            for r in reports:
                n = int(counts.get((ep, r['name']), 0))
                if n:
                    any_rows = True
                    print(f"  [{r['name']}] {n} rows")
            if not any_rows:
                print("  (no data this window)")

print(f"\n--- Run summary ---")
print(f"Windows processed: {len(windows)}")
print(f"Total rows pushed: {total_rows_pushed}")
print(f"Reports failed: {len(failures)} / {len(windows) * len(reports)}")


In [0]:
# ============================================================
# REBUILD "Processed Data (15mins)" from the Data tab
# Runs AFTER this window's rows are appended above, so the rebuild always
# includes them. Pivot: group raw Data rows by (Date Time Range, Hour, Bonus).
#   D:L = sum(StandardHours) per area / divisor ; M = total ;
#   N:V = sum(Quantity) per area for that area's volume event(s).
# Databricks is the sole writer of this tab (Apps Script no longer processes).
# ============================================================
from datetime import datetime
from collections import OrderedDict

def _f(x):
    try:
        return float(str(x).replace(',', '').strip())
    except Exception:
        return 0.0

def _parse_start(dtr):
    try:
        return datetime.strptime(dtr.split(' - ')[0].strip(), '%d/%m/%Y %H:%M')
    except Exception:
        return datetime.max

area_index = {a["report"]: i for i, a in enumerate(PROC_AREAS)}
n_areas = len(PROC_AREAS)

all_data = ws.get_all_values()            # ws = Data worksheet (from earlier cell)
data_rows = all_data[1:] if all_data else []

# Data cols: A=Date B=Hour C=Bonus D=EventType E=Attribute F=Qty G=StdHours
#            H=SMV I=Week J=DateTimeRange K=ReportName
groups = OrderedDict()
for r in data_rows:
    if len(r) < 11:
        continue
    hour = r[1].strip()
    if hour == '':                        # skip "(no data this window)" placeholders
        continue
    ai = area_index.get(r[10].strip())    # Report Name
    if ai is None:
        continue
    # .upper() as well as .strip(): the SQL now canonicalises case on the way
    # in, but this pivot reads the whole Data tab, including rows written
    # before that change and rows added by hand. Folding here too means one
    # person is one row however their code was cased when it landed.
    key = (r[9].strip(), hour, r[2].strip().upper())   # (Date Time Range, Hour, Bonus)
    g = groups.get(key)
    if g is None:
        g = {"std": [0.0] * n_areas, "vol": [0.0] * n_areas}
        groups[key] = g
    g["std"][ai] += _f(r[6])              # Total_StandardHours (all event types)
    if r[3].strip() in PROC_AREAS[ai]["vol_events"]:
        g["vol"][ai] += _f(r[5])          # Total_Quantity (volume event only)

out = []
for (dtr, hour, bonus), g in groups.items():
    std = [s / divisor for s in g["std"]]
    try:
        hour_out = int(hour)
    except ValueError:
        hour_out = hour
    row = [dtr, hour_out, bonus] + std + [sum(std)] + g["vol"]
    out.append((_parse_start(dtr), bonus, row))

out.sort(key=lambda t: (t[0], t[1]))
proc_values = [t[2] for t in out]

proc_ws = sh.worksheet("Processed Data (15mins)")

# Write first, then clear only what is left dangling past the new end.
#
# Clearing first left the tab EMPTY for the whole round trip of the update
# call, and the dashboard polls every minute - so it periodically read a blank
# sheet and showed a blank dashboard. Worse, if the update then failed, the
# tab stayed empty until the next run fifteen minutes later. Writing over the
# top means the tab always holds a complete set of rows: the previous one, or
# this one, never neither.
#
# RAW for the same reason as the Data append above - column C here is the
# bonus code, and USER_ENTERED mangled it identically.
old_last_row = len(proc_ws.col_values(1))       # includes the header row
new_last_row = len(proc_values) + 1

# Adding a work area widens the tab. A sheet whose grid is still the old
# width rejects the wider write outright ("exceeds grid limits"), so grow
# it first; this is a no-op on every run that does not add a column.
if proc_ws.col_count < len(PROC_HEADER):
    proc_ws.resize(cols=len(PROC_HEADER))

# Header and rows go in ONE call, from A1. Written separately there is a
# window where a new header sits over old rows (or the reverse) - and
# since the dashboard maps columns by header NAME, that window mis-reads
# every area after the one that moved rather than failing visibly.
proc_ws.update("A1", [PROC_HEADER] + proc_values, value_input_option="RAW")

if old_last_row > new_last_row:
    proc_ws.batch_clear([f"A{new_last_row + 1}:{PROC_LAST_COL}{old_last_row}"])

print(f"Processed Data (15mins) rebuilt: {len(proc_values)} rows")

In [0]:
if failures:
    try:
        fail_ws = sh.worksheet("Failures")
        fail_existing = fail_ws.col_values(1)
        fail_next_row = len(fail_existing) + 1
    except gspread.exceptions.WorksheetNotFound:
        fail_ws = sh.add_worksheet(title="Failures", rows=1000, cols=4)
        fail_ws.update('A1', [['Window', 'Report', 'Error']])
        fail_next_row = 2

    fail_rows = [[f['window'], f['report'], f['error']] for f in failures]
    fail_ws.update(f'A{fail_next_row}', fail_rows, value_input_option='USER_ENTERED')

    raise Exception(f"{len(failures)} report(s) failed this run: {[f['report'] for f in failures]}")